# Notebook 33 — What does each MJO latent encode? (content analysis + wind-vs-convection)
**Project:** ENSO-BSISO SSL — MJO moisture-constraint experiment
**Author:** Jiayi (jh9141@nyu.edu)

The SSL latent has strong ENSO *displacement* but its angle isn't MJO phase. nb33 asks **what each latent
actually organizes by**, with a key **direct test** added: reconstruct the **u850 field** vs the **OLR field**
from the latent (decoder R2 per field) to settle *wind vs convection* cleanly — not just inferred from a
noisy convection-longitude proxy.

Tools: (1) linear+MLP probes {phase, amplitude, ENSO, conv-longitude, month}; (2) direction-regression;
(3) latent power spectrum; (4) **u850-field vs OLR-field reconstruction R2** (direct wind-vs-convection);
(5) harder targets {MC-crossing, life-cycle, Rossby-Kelvin}; (6) **multi-latent comparison table**.

All latents on the band-passed axis (`X_MJO_bp20_90`) row-align directly. Primary = SSL-nb15; others added
via the `LATENTS` dict.

---

## Cell 1 — Load fields, targets, latents, helpers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, r2_score
from scipy.signal import welch

PROJECT_DIR='/content/drive/MyDrive/BSISO_SSL_Project'; MJO_DIR=f'{PROJECT_DIR}/MJO'
PROC=f'{MJO_DIR}/data/processed'; OUT=f'{MJO_DIR}/moisture_constraints/results/latent_content'
os.makedirs(OUT, exist_ok=True)

labels=pd.read_csv(f'{PROC}/labels_aligned_mjo_bp20_90.csv', parse_dates=['date'])
X_bp=np.load(f'{PROC}/X_MJO_bp20_90.npy'); lons=np.load(f'{PROC}/longitudes_mjo.npy')
M=len(labels); dates=pd.DatetimeIndex(labels['date']).normalize()
u850=X_bp[:,0,0,:].astype(np.float32); olr=X_bp[:,1,0,:].astype(np.float32)
phase=labels['phase'].values.astype(int); amp=labels['amplitude'].values
enso=labels['enso_category'].values; weak=labels['weak_mjo'].values.astype(bool)
active=(~weak)&(amp>=1.0); enso_num=np.array([{'El Nino':1.,'Neutral':0.,'La Nina':-1.}[c] for c in enso])
month=dates.month.values; conv_lon=lons[np.argmin(olr,axis=1)].astype(float); ph_rad=(phase-1)/8.*2*np.pi
years=dates.year.values; is_val=np.isin(years, sorted(np.unique(years))[::5]); tr=~is_val; va=is_val

# candidate latents (all on the bp axis -> row-aligned). Missing files are skipped.
LATENTS={
 'SSL-nb15':   f'{MJO_DIR}/results/ssl/embeddings.npy',
 'aux2d':      f'{MJO_DIR}/moisture_constraints/results/aux2d/embeddings.npy',
 'aux2d_rebal':f'{MJO_DIR}/moisture_constraints/results/aux2d_rebal/embeddings.npy',
 'aux3d':      f'{MJO_DIR}/moisture_constraints/results/aux3d/embeddings.npy',
 'NSV-7D':     f'{MJO_DIR}/nsv/embeddings_v.npy',
 'Barlow-D3':  f'{MJO_DIR}/barlow/D3/embeddings_z7.npy',
 'Barlow-D7':  f'{MJO_DIR}/barlow/embeddings_z7.npy',
}
def load_latent(name):
    p=LATENTS[name]
    if not os.path.exists(p): return None
    Z=np.load(p).astype(np.float32)
    return Z if (Z.ndim==2 and Z.shape[0]==M and Z.shape[1]>=2) else None

# ---- metric helpers (standardize on train) ----
def fieldrecon_r2(Zs, field, mask):
    a=tr&mask; b=va&mask
    r=Ridge(alpha=1.0).fit(Zs[a], field[a])
    return float(r2_score(field[b], r.predict(Zs[b]), multioutput='variance_weighted'))
def clf_acc(Zs, y, mask, bal=False):
    a=tr&mask; b=va&mask
    c=LogisticRegression(max_iter=2000, class_weight=('balanced' if bal else None)).fit(Zs[a],y[a])
    s=balanced_accuracy_score if bal else accuracy_score
    return float(s(y[b], c.predict(Zs[b])))
def reg_r2(Zs, y, mask):
    a=tr&mask&np.isfinite(y); b=va&mask&np.isfinite(y)
    return float(r2_score(y[b], Ridge(alpha=1.0).fit(Zs[a],y[a]).predict(Zs[b])))
def mjo_band_frac(Z):
    pc=PCA(1).fit_transform(Z-Z.mean(0))[:,0]; pco=pc[np.argsort(dates.values)]
    f,P=welch(pco-pco.mean(), fs=1.0, nperseg=min(512,len(pco)), detrend='linear'); f,P=f[1:],P[1:]
    band=(1/f>=30)&(1/f<=90); return float(P[band].sum()/P.sum())
print(f'M={M} active={int(active.sum())}; latents available:',
      [k for k in LATENTS if os.path.exists(LATENTS[k])])

## Cell 2 — Detailed probes for the primary latent (SSL-nb15): linear vs MLP

In [ ]:
NAME='SSL-nb15'; Z=load_latent(NAME); assert Z is not None, 'SSL embeddings missing'
sc=StandardScaler().fit(Z[tr]); Zs=sc.transform(Z)
def clf_probe(y, mask, bal=False):
    a=tr&mask; b=va&mask
    lin=LogisticRegression(max_iter=2000, class_weight=('balanced' if bal else None)).fit(Zs[a],y[a])
    mlp=MLPClassifier(hidden_layer_sizes=(64,32),max_iter=400,random_state=0).fit(Zs[a],y[a])
    s=balanced_accuracy_score if bal else accuracy_score
    return float(s(y[b],lin.predict(Zs[b]))), float(s(y[b],mlp.predict(Zs[b]))), int(b.sum())
def reg_probe(y, mask):
    a=tr&mask&np.isfinite(y); b=va&mask&np.isfinite(y)
    lin=Ridge(1.0).fit(Zs[a],y[a]); mlp=MLPRegressor((64,32),max_iter=600,random_state=0).fit(Zs[a],y[a])
    return float(r2_score(y[b],lin.predict(Zs[b]))), float(r2_score(y[b],mlp.predict(Zs[b]))), int(b.sum())
rows=[]
l,m,n=clf_probe(phase,active);            rows.append(['MJO phase (8-cls acc)',round(1/8,3),round(l,3),round(m,3),n])
l,m,n=reg_probe(amp,np.ones(M,bool));     rows.append(['amplitude (R2)',0.0,round(l,3),round(m,3),n])
l,m,n=clf_probe(enso,np.ones(M,bool),True);rows.append(['ENSO (3-cls bal)',round(1/3,3),round(l,3),round(m,3),n])
l,m,n=reg_probe(conv_lon,active);         rows.append(['conv longitude (R2)',0.0,round(l,3),round(m,3),n])
l,m,n=clf_probe(month,np.ones(M,bool));   rows.append(['month (12-cls acc)',round(1/12,3),round(l,3),round(m,3),n])
probe_df=pd.DataFrame(rows,columns=['target','baseline','linear','MLP','n_val']); print(probe_df.to_string(index=False))
probe_df.to_csv(f'{OUT}/{NAME}_probes.csv',index=False)
fig,ax=plt.subplots(figsize=(11,4.2)); xs=np.arange(len(rows))
ax.bar(xs-0.2,probe_df['linear'],0.38,label='linear',color='#1f77b4'); ax.bar(xs+0.2,probe_df['MLP'],0.38,label='MLP',color='#ff7f0e')
ax.plot(xs,probe_df['baseline'],'k_',ms=18,label='chance'); ax.set_xticks(xs); ax.set_xticklabels(probe_df['target'],rotation=20,ha='right',fontsize=8)
ax.set_ylabel('acc / R2'); ax.set_title(f'{NAME}: linear vs nonlinear decodability'); ax.legend()
plt.tight_layout(); p=f'{OUT}/{NAME}_probes.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 3 — DIRECT wind-vs-convection test: reconstruct u850 vs OLR field from the latent

Linear decoder `z -> field(180 lon)`, R2 on the val set. If **u850 R2 >> OLR R2**, the latent carries the
low-level *wind* structure more than the *convection* (OLR) — settling the wind-vs-convection question
directly (not via the noisy convection-longitude proxy).

In [ ]:
rec=[]
for name in LATENTS:
    Z=load_latent(name)
    if Z is None: continue
    Zsc=StandardScaler().fit(Z[tr]).transform(Z)
    ru=fieldrecon_r2(Zsc,u850,active); ro=fieldrecon_r2(Zsc,olr,active)
    rec.append([name, Z.shape[1], round(ru,3), round(ro,3), round(ru-ro,3)])
rec_df=pd.DataFrame(rec,columns=['latent','dim','u850_recon_R2','OLR_recon_R2','wind_minus_conv'])
print(rec_df.to_string(index=False)); rec_df.to_csv(f'{OUT}/field_reconstruction.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4)); xs=np.arange(len(rec_df))
ax.bar(xs-0.2,rec_df['u850_recon_R2'],0.38,label='u850 (wind)',color='#1f77b4')
ax.bar(xs+0.2,rec_df['OLR_recon_R2'],0.38,label='OLR (convection)',color='#d62728')
ax.set_xticks(xs); ax.set_xticklabels(rec_df['latent'],rotation=20,ha='right',fontsize=8)
ax.set_ylabel('field reconstruction R2'); ax.set_title('Direct wind-vs-convection: which field does each latent encode?'); ax.legend()
plt.tight_layout(); p=f'{OUT}/field_reconstruction.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 4 — Direction-regression + power spectrum (SSL primary)

In [ ]:
Zc=Z=load_latent('SSL-nb15'); Zc=Zc-Zc.mean(0)
def direction(y,mask):
    m=mask&np.isfinite(y); r=Ridge(1.0).fit(Zc[m],y[m]); w=r.coef_[:2]; return w/(np.linalg.norm(w)+1e-9), float(r2_score(y[m],r.predict(Zc[m])))
dirs={k:direction(v,msk) for k,(v,msk) in {'amplitude':(amp,np.ones(M,bool)),'ENSO':(enso_num,np.ones(M,bool)),
      'conv_lon':(conv_lon,active),'phase_cos':(np.cos(ph_rad),active),'phase_sin':(np.sin(ph_rad),active)}.items()}
for k,(w,r2) in dirs.items(): print(f'{k:10s} dir {w.round(2)} R2={r2:.3f}')
order=np.argsort(dates.values); pc=PCA(1).fit_transform(load_latent('SSL-nb15')-load_latent('SSL-nb15').mean(0))[order,0]
f,P=welch(pc-pc.mean(),fs=1.0,nperseg=512,detrend='linear'); f,P=f[1:],P[1:]
fig,ax=plt.subplots(1,2,figsize=(13,4)); ax[0].plot(1/f,P/P.max()); ax[0].axvspan(30,90,color='g',alpha=.1)
ax[0].set_xscale('log'); ax[0].set_xlim(8,400); ax[0].set_xlabel('period (d)'); ax[0].set_title('SSL PC1 spectrum')
sc_=np.percentile(np.abs(Zc),95); col={'amplitude':'g','ENSO':'r','conv_lon':'purple','phase_cos':'navy','phase_sin':'teal'}
ax[1].scatter(Zc[active][:3000,0],Zc[active][:3000,1],s=4,alpha=.1,color='gray')
for k,(w,r2) in dirs.items(): ax[1].arrow(0,0,w[0]*sc_,w[1]*sc_,color=col[k],lw=2,head_width=sc_*.04); ax[1].text(w[0]*sc_,w[1]*sc_,f'{k}\n{r2:.2f}',color=col[k],fontsize=7)
ax[1].set_aspect('equal'); ax[1].set_title('physical directions in latent plane')
plt.tight_layout(); p=f'{OUT}/SSL-nb15_directions_spectrum.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 5 — Multi-latent comparison table (key metrics)

In [ ]:
# event labels for R-K (per-day) reused across latents
west=np.max(u850,axis=1); east=np.max(-u850,axis=1); rk=west/np.maximum(east,1e-6)
comp=[]
for name in LATENTS:
    Z=load_latent(name)
    if Z is None: print('[skip]',name); continue
    Zsc=StandardScaler().fit(Z[tr]).transform(Z)
    comp.append({'latent':name,'dim':Z.shape[1],
        'phase_acc':round(clf_acc(Zsc,phase,active),3),
        'amp_R2':round(reg_r2(Zsc,amp,np.ones(M,bool)),3),
        'ENSO_bal':round(clf_acc(Zsc,enso,np.ones(M,bool),bal=True),3),
        'convlon_R2':round(reg_r2(Zsc,conv_lon,active),3),
        'RK_R2':round(reg_r2(Zsc,rk,active),3),
        'u850rec_R2':round(fieldrecon_r2(Zsc,u850,active),3),
        'OLRrec_R2':round(fieldrecon_r2(Zsc,olr,active),3),
        'MJOband_frac':round(mjo_band_frac(Z),3)})
comp_df=pd.DataFrame(comp); print(comp_df.to_string(index=False)); comp_df.to_csv(f'{OUT}/latent_comparison.csv',index=False)
print('\nbaselines: phase 0.125, ENSO 0.333; R2/recon 0 = none, frac 0..1 (1=all MJO-band)')

## Cell 5b — Add NSV-7D (lag-prediction v-space, date-aligned)

NSV lives on the lag-prediction axis as train/val splits (`v_train.npy`/`v_val.npy` in `state_vars_lag10`,
`d_hat=7`), aligned by `dates_t.npy` + `train_mask.npy` — so it needs recombination + date-alignment to the
bp axis before the same metrics can be computed. Skips gracefully if NSV files are absent.

In [ ]:
NSV_DIR=f'{MJO_DIR}/nsv'; LATENT_DIR=f'{NSV_DIR}/latents_lag10'; STATE_DIR=f'{NSV_DIR}/state_vars_lag10'
def add_nsv():
    need=[f'{STATE_DIR}/v_train.npy',f'{STATE_DIR}/v_val.npy',f'{LATENT_DIR}/train_mask.npy',f'{LATENT_DIR}/dates_t.npy']
    if not all(os.path.exists(q) for q in need):
        print('[skip] NSV-7D files missing'); return None
    try:
        vtr=np.load(need[0]).astype(np.float32); vva=np.load(need[1]).astype(np.float32)
        tmask=np.load(need[2]).astype(bool); dts=pd.DatetimeIndex(np.load(need[3])).normalize()
        if len(dts)!=len(tmask) or tmask.sum()!=len(vtr) or (~tmask).sum()!=len(vva):
            print('[skip] NSV-7D shape mismatch'); return None
        V=np.zeros((len(dts), vtr.shape[1]),np.float32); V[tmask]=vtr; V[~tmask]=vva   # recombine to dates_t order
        bprow={d:i for i,d in enumerate(dates)}; keep=np.array([d in bprow for d in dts])
        bi=np.array([bprow[d] for d in dts[keep]]); Vk=V[keep]
        if len(bi)<500: print('[skip] NSV-7D <500 aligned days'); return None
        yr=dates.year.values[bi]; isv=np.isin(yr, sorted(np.unique(dates.year.values))[::5])
        a=~isv; b=isv; act=active[bi]
        Zsc=StandardScaler().fit(Vk[a]).transform(Vk)
        def acc(y,msk,bal=False):
            c=LogisticRegression(max_iter=2000,class_weight=('balanced' if bal else None)).fit(Zsc[a&msk],y[a&msk])
            s=balanced_accuracy_score if bal else accuracy_score; return float(s(y[b&msk],c.predict(Zsc[b&msk])))
        def r2(y,msk):
            m1=a&msk&np.isfinite(y); m2=b&msk&np.isfinite(y)
            return float(r2_score(y[m2], Ridge(1.0).fit(Zsc[m1],y[m1]).predict(Zsc[m2])))
        def frec(F,msk):
            Fb=F[bi]; m1=a&msk; m2=b&msk
            return float(r2_score(Fb[m2], Ridge(1.0).fit(Zsc[m1],Fb[m1]).predict(Zsc[m2]), multioutput='variance_weighted'))
        west=np.max(u850,1); east=np.max(-u850,1); rkf=(west/np.maximum(east,1e-6))[bi]
        ph=phase[bi]; am=amp[bi]; en=enso[bi]; cl=conv_lon[bi]; ones=np.ones(len(bi),bool)
        pco=PCA(1).fit_transform(Vk-Vk.mean(0))[np.argsort(dates.values[bi]),0]
        f,P=welch(pco-pco.mean(),fs=1.0,nperseg=min(512,len(pco)),detrend='linear'); f,P=f[1:],P[1:]; bd=(1/f>=30)&(1/f<=90)
        return {'latent':'NSV-7D','dim':int(Vk.shape[1]),'phase_acc':round(acc(ph,act),3),
                'amp_R2':round(r2(am,ones),3),'ENSO_bal':round(acc(en,ones,True),3),'convlon_R2':round(r2(cl,act),3),
                'RK_R2':round(r2(rkf,act),3),'u850rec_R2':round(frec(u850,act),3),'OLRrec_R2':round(frec(olr,act),3),
                'MJOband_frac':round(float(P[bd].sum()/P.sum()),3)}
    except Exception as e:
        print('[skip] NSV-7D error:', e); return None
nsv_row=add_nsv()
if nsv_row:
    print('NSV-7D:', nsv_row)
    comp_df=pd.concat([comp_df, pd.DataFrame([nsv_row])], ignore_index=True)
    comp_df.to_csv(f'{OUT}/latent_comparison.csv',index=False); print('\n'+comp_df.to_string(index=False))

## Cell 6 — Hard targets (SSL) + summary

In [ ]:
Zs=StandardScaler().fit(load_latent('SSL-nb15')[tr]).transform(load_latent('SSL-nb15'))
runs=[]; i=0
while i<M:
    if active[i]:
        j=i
        while j+1<M and active[j+1] and int((dates[j+1]-dates[j]).days)==1: j+=1
        if j-i+1>=20: runs.append((i,j))
        i=j+1
    else: i+=1
mc=np.full(M,-1); lc=np.full(M,-1)
for a,b in runs:
    cl=conv_lon[a:b+1]; th=max(3,(b-a)//3)
    mc[a:b+1]=1 if (np.nanmin(cl[:th])<100 and np.nanmax(cl[-th:])>150) else 0
    L=b-a+1; t=L//3; lc[a:a+t]=0; lc[a+t:a+2*t]=1; lc[a+2*t:b+1]=2
hard=[]
mmc=mc>=0
if mmc.sum()>100 and len(np.unique(mc[mmc]))==2:
    base=np.bincount(mc[mmc&va]).max()/ (mmc&va).sum(); hard.append(['MC-crossing acc',round(base,3),round(clf_acc(Zs,mc,mmc),3)])
hard.append(['life-cycle (3-cls)',round(1/3,3),round(clf_acc(Zs,lc,lc>=0),3)])
hard.append(['Rossby-Kelvin R2',0.0,round(reg_r2(Zs,rk,active),3)])
hard_df=pd.DataFrame(hard,columns=['target','baseline','SSL']); print(hard_df.to_string(index=False))
json.dump({'probes':probe_df.to_dict('records'),'directions':{k:{'dir':w.tolist(),'R2':r2} for k,(w,r2) in dirs.items()},
           'comparison':comp_df.to_dict('records'),'field_recon':rec_df.to_dict('records'),'hard':hard_df.to_dict('records')},
          open(f'{OUT}/latent_content_summary.json','w'),indent=2,default=float)
print('\nSaved', f'{OUT}/latent_content_summary.json')
print('Read: u850rec_R2 >> OLRrec_R2 => wind-encoding; MJOband_frac high => intraseasonal (not slow);')
print('phase_acc ~2x chance + amp/convlon ~0 => partial wind-phase, not state/amplitude/location.')

---
## Done!
`MJO/moisture_constraints/results/latent_content/`: `latent_comparison.csv`, `field_reconstruction.csv/png`,
`SSL-nb15_probes`, `_directions_spectrum`, `latent_content_summary.json`.

The **field_reconstruction** table is the direct wind-vs-convection answer; the **comparison** table shows
whether the moisture-aux / NSV / Barlow objectives shift the encoding (toward convection / amplitude /
slow-envelope) vs the plain SSL wind-phase.

---
*DDCS Project | jh9141@nyu.edu*